# First Title
## Second Title
- item 1
- item 2
- item 3
**abcde**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('RFM_Raw.csv')
df.head()

,Date,InvoNo,CustomerKey,Amount
0,2018/01/01,JA1803995,A0620,411.400
1,2018/01/01,JA1805774,A1336,658.240
2,2018/01/01,JA1801197,A0786,1892.352
3,2018/01/02,JA1805224,A0937,411.400
4,2018/01/02,JA1802973,A1371,411.400


In [4]:
import pandas as pd

# 假設你的原始 DataFrame 名稱為 df
# 欄位包含：CustomerKey、Date、InvoNo、Amount

# 1. 確保Date欄位為 datetime 型態，以便進行Date相減
df['Date'] = pd.to_datetime(df['Date'])

# 2. 設定基準Date（即題目中的「Date欄最大Date」，通常是取整個資料集裡最新的那天作為參考點）
# 如果你是要跟「今天」比對，可以改成 pd.Timestamp('today')
reference_date = df['Date'].max()

# 3. 使用 groupby 進行聚合運算
df_rfm = (
    df.groupby('CustomerKey')
    .agg(
        F=('InvoNo', 'count'),  # InvoNo計數
        M=('Amount', 'sum'),  # Amount加總
        # 該客戶的最大Date 與 基準Date的差值（天數）
        # .dt.days 可以將時間差（Timedelta）轉換為整數天數
        R=('Date', lambda x: (reference_date - x.max()).days),
    )
    .reset_index()
)

# 4. 將CustomerKey欄位重新命名為 Customer
df_rfm = df_rfm.rename(columns={'CustomerKey': 'Customer'})

# 調整欄位順序為 Customer, F, M, R (非必要，但排版好看)
df_rfm = df_rfm[['Customer', 'F', 'M', 'R']]

# 檢視結果
print(df_rfm)

    Customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]


In [10]:
df_rfm.to_csv('RFM_Data.csv',index=False)